# Demo AI Chatbot (Groq)
Notebook ini mendemokan chatbot sederhana menggunakan API Groq.

## 1) Install Dependency
Jalankan sekali jika package belum terpasang.

In [1]:
# %pip install -q openai

## 2) Set API Key
Simpan key ke environment variable `GROQ_API_KEY` agar tidak hardcode di notebook.

In [2]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Masukkan GROQ_API_KEY: ")

print("GROQ_API_KEY siap dipakai.")

GROQ_API_KEY siap dipakai.


## 3) Inisialisasi Client dan Fungsi Chatbot

In [11]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

SYSTEM_PROMPT = "Kamu adalah asisten pembelajaran data science yang ramah dan ringkas."

def ask_chatbot(user_message: str, model: str | None = None) -> str:
    preferred_models = [
        "llama-3.3-70b-versatile",
        "llama-3.1-8b-instant",
        "meta-llama/llama-4-scout-17b-16e-instruct",
        "meta-llama/llama-4-maverick-17b-128e-instruct"
    ]

    if model:
        candidate_models = [model]
    else:
        available_ids = [m.id for m in client.models.list().data]
        prioritized = [m for m in preferred_models if m in available_ids]
        remaining = [
            m for m in available_ids
            if m not in prioritized and all(x not in m.lower() for x in ["whisper", "tts", "transcribe"])
        ]
        candidate_models = prioritized + remaining

    if not candidate_models:
        raise RuntimeError("Tidak ada model chat yang tersedia untuk akun Groq ini.")

    last_error = None
    for m in candidate_models:
        if not m:
            continue
        try:
            response = client.chat.completions.create(
                model=m,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.3
            )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            # Lanjut ke model berikutnya jika model tidak tersedia/decommissioned.
            if any(msg in str(e).lower() for msg in ["model_not_found", "does not exist", "decommissioned", "does not support chat completions", "model_terms_required", "requires terms acceptance"]):
                continue
            raise

    raise RuntimeError(f"Semua kandidat model gagal. Error terakhir: {last_error}")

## 4) Demo Tanya Jawab

In [12]:
pertanyaan = "Jelaskan perbedaan klasifikasi dan clustering dalam 3 poin singkat."
jawaban = ask_chatbot(pertanyaan)

print("Pertanyaan:")
print(pertanyaan)
print("\nJawaban Chatbot:")
print(jawaban)

Pertanyaan:
Jelaskan perbedaan klasifikasi dan clustering dalam 3 poin singkat.

Jawaban Chatbot:
**Perbedaan Klasifikasi vs. Clustering (3 poin singkat)**  

1. **Supervisi**  
   - *Klasifikasi*: Memerlukan data berlabel (supervised learning). Model belajar dari contoh “fitur → label”.  
   - *Clustering*: Tidak memerlukan label (unsupervised learning). Algoritma mencari pola struktur data secara mandiri.  

2. **Tujuan Output**  
   - *Klasifikasi*: Menghasilkan kelas diskrit yang telah ditentukan (misalnya spam / bukan spam).  
   - *Clustering*: Mengelompokkan data ke dalam grup yang mirip, tanpa kelas pra‑definisi (misalnya segmentasi pelanggan).  

3. **Evaluasi**  
   - *Klasifikasi*: Dapat diukur dengan akurasi, precision, recall, F‑score karena ada label “kebenaran”.  
   - *Clustering*: Dievaluasi dengan metrik internal (silhouette, inertia) atau eksternal bila ada label referensi (Adjusted Rand Index), karena tidak ada “jawaban” pasti.  


## 5) Mini Chat Loop (Opsional)
Ketik `exit` untuk berhenti.

In [13]:
while True:
    user_text = input("Anda: ")
    if user_text.strip().lower() in {"exit", "quit", "keluar"}:
        print("Sesi selesai.")
        break

    bot_text = ask_chatbot(user_text)
    print("Bot:", bot_text)

Bot: **Data Analytics** (Analisis Data) adalah proses memeriksa, membersihkan, mentransformasi, dan memodelkan data dengan tujuan untuk menemukan informasi yang berguna, menarik kesimpulan, dan mendukung pengambilan keputusan.

Secara sederhana, ini adalah cara mengubah **data mentah** (angka, teks, log) menjadi **wawasan (insight)** yang bisa ditindaklanjuti.

### 3 Jenis Utama Data Analytics:
1. **Descriptive Analytics** (Deskriptif): *Apa yang terjadi?*  
   - Contoh: Laporan penjualan bulan lalu.
2. **Diagnostic Analytics** (Diagnostik): *Mengapa itu terjadi?*  
   - Contoh: Menemukan penyebab penurunan penjualan.
3. **Predictive Analytics** (Prediktif): *Apa yang mungkin terjadi selanjutnya?*  
   - Contoh: Memprediksi permintaan produk untuk bulan depan.
4. **Prescriptive Analytics** (Preskriptif): *Apa yang harus dilakukan?*  
   - Contoh: Rekomendasi strategi diskon untuk meningkatkan penjualan.

### Contoh Penerapan:
- **E-commerce**: Rekomendasi produk ("Pelanggan yang membel

## 6) RAQ/RAG Sederhana: Upload PDF + Tanya Jawab
Bagian ini menambahkan alur sederhana untuk:
1. Upload file PDF
2. Ekstrak isi PDF
3. Buat index TF-IDF untuk retrieval konteks
4. Chatloop tanya jawab berdasarkan isi PDF

Jalankan cell dari atas ke bawah.

In [14]:
# %pip install -q pypdf ipywidgets

In [ ]:
from io import BytesIO
import re
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    return text


def _chunk_text(text: str, chunk_size: int = 800, overlap: int = 120):
    words = text.split()
    if not words:
        return []

    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:
            chunks.append(" ".join(chunk_words))
    return chunks


def _extract_pdf_text_from_bytes(pdf_bytes: bytes) -> str:
    reader = PdfReader(BytesIO(pdf_bytes))
    pages = []
    for page in reader.pages:
        pages.append(_clean_text(page.extract_text()))
    return "\n".join([p for p in pages if p])


uploader = widgets.FileUpload(accept=".pdf", multiple=False, description="Upload PDF")
display(uploader)
print("Klik tombol Upload PDF, pilih 1 file PDF, lalu jalankan cell berikutnya untuk membuat index.")

In [ ]:
PDF_CHUNKS = []
PDF_VECTORIZER = None
PDF_MATRIX = None


def build_pdf_index_from_upload(upload_widget):
    global PDF_CHUNKS, PDF_VECTORIZER, PDF_MATRIX

    if not upload_widget.value:
        raise ValueError("Belum ada file PDF yang di-upload.")

    item = next(iter(upload_widget.value.values()))
    file_name = item.get("name", "dokumen.pdf")
    pdf_bytes = item["content"]

    full_text = _extract_pdf_text_from_bytes(pdf_bytes)
    if not full_text.strip():
        raise ValueError("Isi PDF kosong atau tidak bisa diekstrak.")

    PDF_CHUNKS = _chunk_text(full_text, chunk_size=800, overlap=120)
    PDF_VECTORIZER = TfidfVectorizer()
    PDF_MATRIX = PDF_VECTORIZER.fit_transform(PDF_CHUNKS)

    print(f"Index siap dari '{file_name}'. Jumlah chunk: {len(PDF_CHUNKS)}")


def ask_pdf_chatbot(question: str, top_k: int = 3) -> str:
    if not PDF_CHUNKS or PDF_VECTORIZER is None or PDF_MATRIX is None:
        raise RuntimeError("Index PDF belum dibuat. Jalankan build_pdf_index_from_upload(uploader) dulu.")

    q_vec = PDF_VECTORIZER.transform([question])
    scores = cosine_similarity(q_vec, PDF_MATRIX)[0]
    top_indices = scores.argsort()[-top_k:][::-1]
    context = "\n\n".join([f"[Chunk {i+1}] {PDF_CHUNKS[i]}" for i in top_indices])

    prompt = (
        "Jawab pertanyaan pengguna hanya berdasarkan konteks PDF berikut. "
        "Jika tidak ditemukan, jawab jujur bahwa informasi tidak ada di dokumen.\n\n"
        f"Konteks:\n{context}\n\n"
        f"Pertanyaan: {question}"
    )
    return ask_chatbot(prompt)


build_pdf_index_from_upload(uploader)

print("\nMulai chatloop PDF. Ketik 'exit' untuk berhenti.")
while True:
    q = input("Anda (PDF): ")
    if q.strip().lower() in {"exit", "quit", "keluar"}:
        print("Sesi PDF selesai.")
        break

    ans = ask_pdf_chatbot(q, top_k=3)
    print("Bot (PDF):", ans)
    print("-" * 70)